# ARIAN Phase 4 — Wildfire Prediction & Hypothesis Testing

Trains an XGBoost classifier on historical fire labels + weather features,
then scores the 30-day hourly weather predictions from Notebook 3 to produce
**hourly wildfire risk probabilities** over the same 30-day horizon.

Includes automated hypothesis generation comparing projected conditions
against historical baselines.

**Inputs:**
  - `data/processed/master_daily.parquet`         — historical daily matrix
  - `data/processed/fires_daily.parquet`           — fire labels
  - `data/raw/weather_hourly.parquet`              — hourly historical weather
  - `outputs/phase3_weather_hourly_30d.parquet`    — 30-day hourly forecast (NB 3)
  - `data/reference/static_geography.parquet`      — static features
  - `data/reference/cities.parquet`                — city coordinates

**Outputs:**
  - `outputs/phase4_wildfire_hourly_30d.parquet`   — hourly fire-risk forecast
  - `outputs/phase4_wildfire_scores.csv`           — test-set metrics
  - `outputs/phase4_hypotheses.csv`                — automated hypotheses
  - `outputs/phase4_risk_map.html`                 — Folium risk map
  - `models/phase4_xgb_fire.joblib`                — trained model

## §0 · Setup

In [1]:
%pip install -q xgboost scikit-learn pandas numpy pyarrow tqdm \
    folium plotly joblib scipy 2>/dev/null || \
!pip install -q xgboost scikit-learn pandas numpy pyarrow tqdm \
    folium plotly joblib scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ─── ARIAN — Universal environment & path setup ───────────────────────────
import os, sys
from pathlib import Path

PROJECT_NAME = "ARIAN_Data"

def _detect_project_root() -> Path:
    if os.environ.get("ARIAN_ROOT"):
        return Path(os.environ["ARIAN_ROOT"]).expanduser().resolve()
    if "google.colab" in sys.modules:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        return Path(f"/content/drive/MyDrive/{PROJECT_NAME}")
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data").is_dir() and (cand / "notebooks").is_dir():
            return cand
        if cand.name == PROJECT_NAME and (cand / "data").is_dir():
            return cand
    return here.parent if here.name == "notebooks" else here

ROOT      = _detect_project_root()
DATA      = ROOT / "data"
RAW       = DATA / "raw"
LEGACY    = RAW  / "legacy"
PROCESSED = DATA / "processed"
REFERENCE = DATA / "reference"
OUTPUTS   = ROOT / "outputs"
MODELS    = ROOT / "models"

for p in (RAW, LEGACY, PROCESSED, REFERENCE, OUTPUTS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

print(f"📁 Project root : {ROOT}")

📁 Project root : /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF


In [3]:
# ─── Imports & data loading ────────────────────────────────────────────────
import warnings, json, joblib
from datetime import datetime, timezone
from typing import Dict, List, Tuple
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    precision_recall_curve,
)
import folium
from folium.plugins import HeatMap
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Load historical data ───────────────────────────────────────────────────
daily = pd.read_parquet(PROCESSED / "master_daily.parquet")
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["City", "Date"]).reset_index(drop=True)

hourly_hist = pd.read_parquet(RAW / "weather_hourly.parquet")
hourly_hist["Timestamp"] = pd.to_datetime(hourly_hist["Timestamp"])

# ── Load NB3 weather forecast ─────────────────────────────────────────────
FCST_PATH = OUTPUTS / "phase3_weather_hourly_30d.parquet"
if not FCST_PATH.exists():
    raise FileNotFoundError(f"Missing {FCST_PATH}. Run notebook 03 first.")
weather_fc = pd.read_parquet(FCST_PATH)
weather_fc["Timestamp"] = pd.to_datetime(weather_fc["Timestamp"])

# ── Load static geography ─────────────────────────────────────────────────
static_path = REFERENCE / "static_geography.parquet"
static_df = pd.read_parquet(static_path) if static_path.exists() else pd.DataFrame()

# ── Load cities ────────────────────────────────────────────────────────────
cities_path = REFERENCE / "cities.parquet"
if cities_path.exists():
    cities_ref = pd.read_parquet(cities_path)
    CITIES = {r["City"]: (r["Latitude"], r["Longitude"]) for _, r in cities_ref.iterrows()}
else:
    CITIES = {c: (daily.loc[daily["City"] == c, "Latitude"].iloc[0],
                   daily.loc[daily["City"] == c, "Longitude"].iloc[0])
              for c in daily["City"].unique() if "Latitude" in daily.columns}

WEATHER_FEATURES = [c for c in weather_fc.columns
                    if c not in ("City", "Latitude", "Longitude", "Timestamp")
                    and weather_fc[c].dtype in ("float64", "float32", "int64")]

print(f"Historical daily: {len(daily):,} rows")
print(f"Historical hourly: {len(hourly_hist):,} rows")
print(f"Weather forecast: {len(weather_fc):,} rows")
print(f"Cities: {len(CITIES)}")
print(f"Weather features: {WEATHER_FEATURES}")

Historical daily: 83,392 rows
Historical hourly: 2,001,408 rows
Weather forecast: 11,520 rows
Cities: 16
Weather features: ['Temperature_C', 'Humidity_percent', 'Rain_mm', 'Wind_Speed_kmh', 'Wind_Direction_deg', 'Pressure_hPa', 'Solar_Radiation_Wm2', 'Soil_Temp_C', 'Soil_Moisture']


## §1 · Feature engineering for hourly fire classification

In [5]:
def engineer_hourly_fire_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build fire-relevant features from hourly weather data.
    Must only use columns available in both historical and forecast data."""
    df = df.copy()
    ts = df["Timestamp"]

    # Time features
    df["hour"]      = ts.dt.hour
    df["dayofweek"] = ts.dt.dayofweek
    df["dayofyear"] = ts.dt.dayofyear
    df["month"]     = ts.dt.month
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    df["doy_sin"]   = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
    df["doy_cos"]   = np.cos(2 * np.pi * df["dayofyear"] / 365.25)
    df["is_summer"] = df["month"].isin([6, 7, 8, 9]).astype(int)
    df["is_daytime"] = df["hour"].between(6, 20).astype(int)

    # FWI-lite at hourly resolution
    if "Humidity_percent" in df.columns and "Temperature_C" in df.columns:
        df["FFMC_h"] = (101 - 5.0 * (df["Humidity_percent"] / 10.0)
                        + 0.3 * df["Temperature_C"]
                        - 2.0 * df.get("Rain_mm", 0)).clip(0, 101)
    if "Wind_Speed_kmh" in df.columns and "FFMC_h" in df.columns:
        df["ISI_h"] = df["Wind_Speed_kmh"] * np.exp(0.05 * df["FFMC_h"])

    # Rolling features (per city)
    g = df.groupby("City")
    if "Temperature_C" in df.columns:
        df["temp_roll24_mean"] = g["Temperature_C"].transform(
            lambda s: s.rolling(24, min_periods=1).mean())
        df["temp_roll168_mean"] = g["Temperature_C"].transform(
            lambda s: s.rolling(168, min_periods=1).mean())
    if "Rain_mm" in df.columns:
        df["rain_roll24_sum"] = g["Rain_mm"].transform(
            lambda s: s.rolling(24, min_periods=1).sum())
        df["rain_roll168_sum"] = g["Rain_mm"].transform(
            lambda s: s.rolling(168, min_periods=1).sum())
        df["is_dry_hour"] = (df["Rain_mm"] < 0.1).astype(int)
        df["dry_streak"] = g["is_dry_hour"].transform(
            lambda s: s.groupby((s != s.shift()).cumsum()).cumsum() * s)
    if "Humidity_percent" in df.columns:
        df["humid_roll24_min"] = g["Humidity_percent"].transform(
            lambda s: s.rolling(24, min_periods=1).min())

    return df


# Build training data: merge hourly weather with daily fire labels
fires_daily_path = PROCESSED / "fires_daily.parquet"
if fires_daily_path.exists():
    fires_daily = pd.read_parquet(fires_daily_path)
    fires_daily["Date"] = pd.to_datetime(fires_daily["Date"])
else:
    fires_daily = daily[["City", "Date", "Fire_Occurred"]].copy()

# Merge fire label into hourly (broadcast daily label to all 24 hours)
hist = hourly_hist.copy()
hist["Date"] = hist["Timestamp"].dt.tz_localize(None).dt.normalize()

# Ensure fires_daily Date is also tz-naive
if hasattr(fires_daily["Date"].dt, "tz") and fires_daily["Date"].dt.tz is not None:
    fires_daily["Date"] = fires_daily["Date"].dt.tz_localize(None)

fire_merge = fires_daily[["City", "Date", "Fire_Occurred"]].drop_duplicates()
hist = hist.merge(fire_merge, on=["City", "Date"], how="left")
hist["Fire_Occurred"] = hist["Fire_Occurred"].fillna(0).astype(int)

# Merge static geography
if not static_df.empty:
    static_cols = [c for c in static_df.columns if c != "City" or c == "City"]
    hist = hist.merge(static_df, on="City", how="left", suffixes=("", "_static"))
    # Remove duplicate coordinate columns
    for dup in [c for c in hist.columns if c.endswith("_static")]:
        hist.drop(columns=dup, inplace=True, errors="ignore")

hist = engineer_hourly_fire_features(hist)

# Identify feature columns (everything except identifiers and label)
EXCLUDE = {"City", "Timestamp", "Date", "Fire_Occurred", "Latitude", "Longitude"}
FIRE_FEATURES = [c for c in hist.columns
                 if c not in EXCLUDE and hist[c].dtype in ("float64", "float32", "int64", "int32", "uint8")]

print(f"Hourly training matrix: {hist.shape}")
print(f"Fire features ({len(FIRE_FEATURES)}): {FIRE_FEATURES[:20]}…")
print(f"Fire rate: {hist['Fire_Occurred'].mean()*100:.3f}%")

Hourly training matrix: (2001408, 39)
Fire features (33): ['Temperature_C', 'Humidity_percent', 'Rain_mm', 'Wind_Speed_kmh', 'Wind_Direction_deg', 'Pressure_hPa', 'Solar_Radiation_Wm2', 'Soil_Temp_C', 'Soil_Moisture', 'Elevation', 'Slope', 'Trees_pct', 'Urban_pct', 'Pop_Total', 'hour', 'dayofweek', 'dayofyear', 'month', 'hour_sin', 'hour_cos']…
Fire rate: 10.863%


## §2 · Train XGBoost fire classifier

In [8]:
# Time-respecting split: 85% train, 15% test
SPLIT_DATE = hist["Timestamp"].quantile(0.85)
print(f"Split at {SPLIT_DATE}")

# Fill NaN in feature columns (static geography / rolling warm-up may leave gaps)
for feat in FIRE_FEATURES:
    if hist[feat].isna().any():
        hist[feat] = hist[feat].fillna(0)

train_h = hist[hist["Timestamp"] <= SPLIT_DATE].copy()
test_h  = hist[hist["Timestamp"]  > SPLIT_DATE].copy()

X_train = train_h[FIRE_FEATURES].values
y_train = train_h["Fire_Occurred"].astype(int).values
X_test  = test_h[FIRE_FEATURES].values
y_test  = test_h["Fire_Occurred"].astype(int).values

scale_pos_weight = max((y_train == 0).sum() / max((y_train == 1).sum(), 1), 1.0)
print(f"Train: {X_train.shape} · pos rate {y_train.mean()*100:.3f}%")
print(f"Test:  {X_test.shape} · pos rate {y_test.mean()*100:.3f}%")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# Validation slice for early stopping (last 8% of train)
val_cutoff = train_h["Timestamp"].quantile(0.92)
fit_mask = train_h["Timestamp"] <= val_cutoff
val_mask = train_h["Timestamp"]  > val_cutoff

X_fit = train_h.loc[fit_mask, FIRE_FEATURES].values
y_fit = train_h.loc[fit_mask, "Fire_Occurred"].astype(int).values
X_val = train_h.loc[val_mask, FIRE_FEATURES].values
y_val = train_h.loc[val_mask, "Fire_Occurred"].astype(int).values

print(f"Fit: {X_fit.shape} · Val: {X_val.shape}")

xgb_clf = xgb.XGBClassifier(
    n_estimators=1500, learning_rate=0.05, max_depth=6,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, objective="binary:logistic", eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight, tree_method="hist",
    n_jobs=-1, random_state=42, early_stopping_rounds=50,
)
xgb_clf.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
print(f"Best iteration: {xgb_clf.best_iteration} · Best PR-AUC: {xgb_clf.best_score:.4f}")

# Isotonic calibration — sklearn ≥1.8 removed cv="prefit", use FrozenEstimator
from sklearn.frozen import FrozenEstimator
calibrated = CalibratedClassifierCV(
    FrozenEstimator(xgb_clf), method="isotonic", cv=5)
calibrated.fit(X_val, y_val)
print("✅ Isotonic calibration fitted")

Split at 2024-03-07 04:00:00+00:00
Train: (1701200, 33) · pos rate 10.987%
Test:  (300208, 33) · pos rate 10.161%
scale_pos_weight: 8.10
Fit: (1565104, 33) · Val: (136096, 33)
Best iteration: 25 · Best PR-AUC: 0.4532
✅ Isotonic calibration fitted


## §3 · Evaluation

In [9]:
proba_cal = calibrated.predict_proba(X_test)[:, 1]

# Operational threshold — max F1 on validation
val_proba = calibrated.predict_proba(X_val)[:, 1]
prec_v, rec_v, thr_v = precision_recall_curve(y_val, val_proba)
f1_curve = 2 * prec_v[:-1] * rec_v[:-1] / np.clip(prec_v[:-1] + rec_v[:-1], 1e-9, None)
op_thr = float(thr_v[np.nanargmax(f1_curve)])
print(f"Operational threshold (max-F1 on val): {op_thr:.4f}")

y_pred_op = (proba_cal >= op_thr).astype(int)

def score_model(name, y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    return {
        "model": name,
        "PR_AUC": average_precision_score(y_true, proba),
        "ROC_AUC": roc_auc_score(y_true, proba),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Threshold": threshold,
    }

scores = pd.DataFrame([
    score_model("XGB (cal + op-thr)", y_test, proba_cal, op_thr),
]).round(4)

print("\n=== Test-set performance ===")
print(scores.to_string(index=False))
print(f"\n=== Classification report @ {op_thr:.3f} ===")
print(classification_report(y_test, y_pred_op, target_names=["no_fire", "fire"]))

scores.to_csv(OUTPUTS / "phase4_wildfire_scores.csv", index=False)

# Feature importance
importance = (pd.DataFrame({"feature": FIRE_FEATURES,
                             "gain": xgb_clf.feature_importances_})
              .sort_values("gain", ascending=False).round(4))
print("\n=== Top-15 feature importance ===")
print(importance.head(15).to_string(index=False))
importance.to_csv(OUTPUTS / "phase4_feature_importance.csv", index=False)

# Save model
joblib.dump(calibrated, MODELS / "phase4_xgb_fire.joblib")
manifest = {
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "feature_order": FIRE_FEATURES,
    "operational_threshold": op_thr,
    "test_metrics": {k: float(v) for k, v in scores.iloc[0].items() if k != "model"},
}
(MODELS / "phase4_manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\n✅ Model saved → {MODELS}")

Operational threshold (max-F1 on val): 0.2539

=== Test-set performance ===
             model  PR_AUC  ROC_AUC     F1  Precision  Recall  Threshold
XGB (cal + op-thr)  0.3281    0.799 0.3888     0.3922  0.3854     0.2539

=== Classification report @ 0.254 ===
              precision    recall  f1-score   support

     no_fire       0.93      0.93      0.93    269704
        fire       0.39      0.39      0.39     30504

    accuracy                           0.88    300208
   macro avg       0.66      0.66      0.66    300208
weighted avg       0.88      0.88      0.88    300208


=== Top-15 feature importance ===
         feature   gain
       is_summer 0.4208
   Soil_Moisture 0.0902
       Urban_pct 0.0600
           month 0.0500
humid_roll24_min 0.0490
temp_roll24_mean 0.0451
       Trees_pct 0.0346
       Elevation 0.0306
Humidity_percent 0.0292
       dayofyear 0.0263
       Pop_Total 0.0247
           Slope 0.0219
         doy_sin 0.0213
         doy_cos 0.0195
        hour_cos 

## §4 · Score the 30-day hourly weather forecast

In [11]:
# Prepare the forecast data with the same features used for training
# Prepend last 7 days of history so rolling features have a warm-up window
hist_tail = (hourly_hist.sort_values(["City", "Timestamp"])
             .groupby("City").tail(168))  # 7 days

# Merge static geography into forecast
fc_scored = weather_fc.copy()
if not static_df.empty:
    fc_scored = fc_scored.merge(static_df, on="City", how="left", suffixes=("", "_static"))
    for dup in [c for c in fc_scored.columns if c.endswith("_static")]:
        fc_scored.drop(columns=dup, inplace=True, errors="ignore")

# Also merge static into history tail
ht = hist_tail.copy()
if not static_df.empty:
    ht = ht.merge(static_df, on="City", how="left", suffixes=("", "_static"))
    for dup in [c for c in ht.columns if c.endswith("_static")]:
        ht.drop(columns=dup, inplace=True, errors="ignore")

# Concatenate history tail + forecast, engineer features, then slice forecast only
combined = pd.concat([ht, fc_scored], ignore_index=True)
combined = combined.sort_values(["City", "Timestamp"]).reset_index(drop=True)
combined = engineer_hourly_fire_features(combined)

# Slice back to only the forecast window
fc_start = weather_fc["Timestamp"].min()
scored_df = combined[combined["Timestamp"] >= fc_start].copy()

# Ensure all required features exist; fill missing with 0
for feat in FIRE_FEATURES:
    if feat not in scored_df.columns:
        scored_df[feat] = 0
    scored_df[feat] = scored_df[feat].fillna(0)

X_fc = scored_df[FIRE_FEATURES].values
print(f"Forecast scoring matrix: {X_fc.shape}")

risk_proba = calibrated.predict_proba(X_fc)[:, 1]
scored_df["fire_risk_prob"]  = risk_proba
scored_df["fire_risk_alert"] = (risk_proba >= op_thr).astype(int)
scored_df["risk_class"] = pd.cut(
    scored_df["fire_risk_prob"],
    bins=[-0.01, 0.05, 0.20, 0.50, 1.01],
    labels=["Low", "Moderate", "High", "Extreme"],
)

out_cols = (["City", "Timestamp", "fire_risk_prob", "fire_risk_alert", "risk_class"]
            + [f for f in WEATHER_FEATURES if f in scored_df.columns])
fire_fc = scored_df[out_cols].copy()
fire_fc.to_parquet(OUTPUTS / "phase4_wildfire_hourly_30d.parquet", index=False)

print(f"✅ Hourly wildfire forecast: {len(fire_fc):,} rows "
      f"({fire_fc['City'].nunique()} cities)")
print(f"\nRisk-class distribution:")
print(fire_fc["risk_class"].value_counts())
print(f"\nTop-10 highest-risk hours:")
print(fire_fc.nlargest(10, "fire_risk_prob")[
    ["City", "Timestamp", "fire_risk_prob", "risk_class"]].to_string(index=False))

Forecast scoring matrix: (11520, 33)
✅ Hourly wildfire forecast: 11,520 rows (16 cities)

Risk-class distribution:
risk_class
Low         9057
Moderate    2002
High         461
Extreme        0
Name: count, dtype: int64

Top-10 highest-risk hours:
City                 Timestamp  fire_risk_prob risk_class
Baku 2026-05-07 10:00:00+00:00        0.222792       High
Baku 2026-05-07 11:00:00+00:00        0.222792       High
Baku 2026-05-07 12:00:00+00:00        0.222792       High
Baku 2026-05-07 13:00:00+00:00        0.222792       High
Baku 2026-05-07 14:00:00+00:00        0.222792       High
Baku 2026-05-08 06:00:00+00:00        0.222792       High
Baku 2026-05-08 07:00:00+00:00        0.222792       High
Baku 2026-05-08 08:00:00+00:00        0.222792       High
Baku 2026-05-08 09:00:00+00:00        0.222792       High
Baku 2026-05-08 10:00:00+00:00        0.222792       High


## §5 · Automated hypothesis generation & analysis

In [12]:
from scipy import stats as sp_stats

hypotheses = []

# --- Historical baselines ---
hist_daily = daily.copy()
hist_daily["year"] = hist_daily["Date"].dt.year

# Forecast daily aggregates
fc_daily = fire_fc.copy()
fc_daily["Date"] = fc_daily["Timestamp"].dt.date

# ═══════════════════════════════════════════════════════════════════════════
# Hypothesis 1: Is the projected 30-day period DRIER than last year's
#               same calendar window?
# ═══════════════════════════════════════════════════════════════════════════
if "Rain_mm" in fire_fc.columns:
    fc_rain_by_city = (fc_daily.groupby("City")["Rain_mm"]
                       .sum().reset_index().rename(columns={"Rain_mm": "fc_rain_total"}))
    last_year = hist_daily["year"].max()
    fc_dates  = sorted(fc_daily["Date"].unique())
    cal_start = pd.Timestamp(last_year, fc_dates[0].month, fc_dates[0].day)
    cal_end   = cal_start + pd.Timedelta(days=29)

    ly_rain = (hist_daily[(hist_daily["Date"] >= cal_start) &
                          (hist_daily["Date"] <= cal_end)]
               .groupby("City")["Rain_mm_sum"].sum()
               .reset_index().rename(columns={"Rain_mm_sum": "ly_rain_total"}))
    comp = fc_rain_by_city.merge(ly_rain, on="City", how="inner")
    if not comp.empty:
        comp["drier"] = comp["fc_rain_total"] < comp["ly_rain_total"]
        n_drier = int(comp["drier"].sum())
        avg_diff = float((comp["fc_rain_total"] - comp["ly_rain_total"]).mean())
        hypotheses.append({
            "Hypothesis": "Projected 30-day rainfall vs. same window last year",
            "Result": f"{n_drier}/{len(comp)} cities projected to be drier",
            "Avg_difference": f"{avg_diff:+.1f} mm",
            "Conclusion": "DRIER" if n_drier > len(comp) / 2 else "NOT drier",
        })

# ═══════════════════════════════════════════════════════════════════════════
# Hypothesis 2: Is the projected 30-day period drier than the
#               last-decade average for the same calendar window?
# ═══════════════════════════════════════════════════════════════════════════
if "Rain_mm" in fire_fc.columns and "Rain_mm_sum" in hist_daily.columns:
    decade_rain_rows = []
    for city in fire_fc["City"].unique():
        city_hist = hist_daily[hist_daily["City"] == city]
        decade_vals = []
        for yr in range(max(2012, last_year - 9), last_year + 1):
            try:
                w_start = pd.Timestamp(yr, fc_dates[0].month, fc_dates[0].day)
                w_end   = w_start + pd.Timedelta(days=29)
                total = city_hist[(city_hist["Date"] >= w_start) &
                                  (city_hist["Date"] <= w_end)]["Rain_mm_sum"].sum()
                decade_vals.append(total)
            except Exception:
                pass
        if decade_vals:
            decade_rain_rows.append({
                "City": city,
                "decade_mean_rain": np.mean(decade_vals),
                "decade_std_rain": np.std(decade_vals),
            })
    if decade_rain_rows:
        decade_df = pd.merge(fc_rain_by_city, pd.DataFrame(decade_rain_rows), on="City")
        decade_df["drier_than_decade"] = decade_df["fc_rain_total"] < decade_df["decade_mean_rain"]
        n_drier_dec = int(decade_df["drier_than_decade"].sum())
        hypotheses.append({
            "Hypothesis": "Projected 30-day rainfall vs. last-decade average (same window)",
            "Result": f"{n_drier_dec}/{len(decade_df)} cities projected drier than decade avg",
            "Avg_difference": f"{(decade_df['fc_rain_total'] - decade_df['decade_mean_rain']).mean():+.1f} mm",
            "Conclusion": "DRIER than decade" if n_drier_dec > len(decade_df) / 2 else "NOT drier than decade",
        })

# ═══════════════════════════════════════════════════════════════════════════
# Hypothesis 3: Will predicted wildfire frequency INCREASE compared to the
#               historical fire rate for the same calendar window?
# ═══════════════════════════════════════════════════════════════════════════
if "Fire_Occurred" in hist_daily.columns:
    fc_fire_city = (fc_daily.groupby("City")["fire_risk_alert"]
                    .sum().reset_index().rename(columns={"fire_risk_alert": "fc_fire_hours"}))
    # Historical fire-days per city for same calendar window (averaged over years)
    hist_fire_rows = []
    for city in fire_fc["City"].unique():
        city_hist = hist_daily[hist_daily["City"] == city]
        yearly_counts = []
        for yr in range(max(2012, last_year - 9), last_year + 1):
            try:
                w_start = pd.Timestamp(yr, fc_dates[0].month, fc_dates[0].day)
                w_end   = w_start + pd.Timedelta(days=29)
                n_fire = city_hist[(city_hist["Date"] >= w_start) &
                                   (city_hist["Date"] <= w_end)]["Fire_Occurred"].sum()
                yearly_counts.append(n_fire)
            except Exception:
                pass
        if yearly_counts:
            hist_fire_rows.append({
                "City": city,
                "hist_avg_fire_days": np.mean(yearly_counts),
            })
    if hist_fire_rows:
        fire_comp = pd.merge(fc_fire_city, pd.DataFrame(hist_fire_rows), on="City")
        # Convert fc fire hours to fire-day equivalents (any hour with alert → 1 fire-day)
        fc_fire_days = (fc_daily.groupby("City")["fire_risk_alert"]
                        .apply(lambda s: (s.groupby(fc_daily.loc[s.index, "Date"]).max() > 0).sum())
                        .reset_index().rename(columns={"fire_risk_alert": "fc_fire_days"}))
        fire_comp = fire_comp.merge(fc_fire_days, on="City", how="left")
        fire_comp["fc_fire_days"] = fire_comp["fc_fire_days"].fillna(0)
        fire_comp["increase"] = fire_comp["fc_fire_days"] > fire_comp["hist_avg_fire_days"]
        n_increase = int(fire_comp["increase"].sum())
        hypotheses.append({
            "Hypothesis": "Will wildfire frequency increase vs. historical average?",
            "Result": f"{n_increase}/{len(fire_comp)} cities projected to have more fire days",
            "Avg_difference": f"{(fire_comp['fc_fire_days'] - fire_comp['hist_avg_fire_days']).mean():+.1f} fire-days",
            "Conclusion": "INCREASE expected" if n_increase > len(fire_comp) / 2 else "No general increase expected",
        })

# ═══════════════════════════════════════════════════════════════════════════
# Hypothesis 4: Is the projected temperature HOTTER than last year's
#               same window? (Heat-stress indicator)
# ═══════════════════════════════════════════════════════════════════════════
if "Temperature_C" in fire_fc.columns and "Temperature_C_max" in hist_daily.columns:
    fc_temp = (fc_daily.groupby("City")["Temperature_C"]
               .mean().reset_index().rename(columns={"Temperature_C": "fc_temp_avg"}))
    ly_temp = (hist_daily[(hist_daily["Date"] >= cal_start) &
                          (hist_daily["Date"] <= cal_end)]
               .groupby("City")["Temperature_C_max"].mean()
               .reset_index().rename(columns={"Temperature_C_max": "ly_temp_avg"}))
    temp_comp = fc_temp.merge(ly_temp, on="City", how="inner")
    if not temp_comp.empty:
        temp_comp["hotter"] = temp_comp["fc_temp_avg"] > temp_comp["ly_temp_avg"]
        n_hotter = int(temp_comp["hotter"].sum())
        hypotheses.append({
            "Hypothesis": "Projected temperature vs. same window last year",
            "Result": f"{n_hotter}/{len(temp_comp)} cities projected hotter",
            "Avg_difference": f"{(temp_comp['fc_temp_avg'] - temp_comp['ly_temp_avg']).mean():+.1f} °C",
            "Conclusion": "HOTTER" if n_hotter > len(temp_comp) / 2 else "NOT hotter",
        })

# ═══════════════════════════════════════════════════════════════════════════
# Hypothesis 5: Is projected humidity LOWER (drier) than the historical
#               average for this window? (dehydration / fire risk signal)
# ═══════════════════════════════════════════════════════════════════════════
if "Humidity_percent" in fire_fc.columns and "Humidity_percent_min" in hist_daily.columns:
    fc_hum = (fc_daily.groupby("City")["Humidity_percent"]
              .mean().reset_index().rename(columns={"Humidity_percent": "fc_humid_avg"}))
    hist_hum_rows = []
    for city in fire_fc["City"].unique():
        city_hist = hist_daily[hist_daily["City"] == city]
        vals = []
        for yr in range(max(2012, last_year - 9), last_year + 1):
            try:
                w_start = pd.Timestamp(yr, fc_dates[0].month, fc_dates[0].day)
                w_end   = w_start + pd.Timedelta(days=29)
                avg_h = city_hist[(city_hist["Date"] >= w_start) &
                                  (city_hist["Date"] <= w_end)]["Humidity_percent_min"].mean()
                if not np.isnan(avg_h):
                    vals.append(avg_h)
            except Exception:
                pass
        if vals:
            hist_hum_rows.append({"City": city, "hist_humid_avg": np.mean(vals)})
    if hist_hum_rows:
        hum_comp = fc_hum.merge(pd.DataFrame(hist_hum_rows), on="City")
        hum_comp["drier_air"] = hum_comp["fc_humid_avg"] < hum_comp["hist_humid_avg"]
        n_drier_air = int(hum_comp["drier_air"].sum())
        hypotheses.append({
            "Hypothesis": "Projected humidity vs. decade average (lower = drier air)",
            "Result": f"{n_drier_air}/{len(hum_comp)} cities have lower predicted humidity",
            "Avg_difference": f"{(hum_comp['fc_humid_avg'] - hum_comp['hist_humid_avg']).mean():+.1f} %",
            "Conclusion": "DRIER air expected" if n_drier_air > len(hum_comp) / 2 else "NOT drier air",
        })

# ── Output ─────────────────────────────────────────────────────────────────
hyp_df = pd.DataFrame(hypotheses)
hyp_df.to_csv(OUTPUTS / "phase4_hypotheses.csv", index=False)
print("=" * 80)
print("AUTOMATED HYPOTHESIS REPORT")
print("=" * 80)
for i, row in hyp_df.iterrows():
    print(f"\n🔬 H{i+1}: {row['Hypothesis']}")
    print(f"   📊 {row['Result']}")
    print(f"   📏 Avg difference: {row['Avg_difference']}")
    print(f"   ✅ Conclusion: {row['Conclusion']}")
print(f"\n💾 Saved → {OUTPUTS / 'phase4_hypotheses.csv'}")

AUTOMATED HYPOTHESIS REPORT

🔬 H1: Projected 30-day rainfall vs. last-decade average (same window)
   📊 0/16 cities projected drier than decade avg
   📏 Avg difference: +0.0 mm
   ✅ Conclusion: NOT drier than decade

🔬 H2: Will wildfire frequency increase vs. historical average?
   📊 0/16 cities projected to have more fire days
   📏 Avg difference: -2.0 fire-days
   ✅ Conclusion: No general increase expected

🔬 H3: Projected humidity vs. decade average (lower = drier air)
   📊 0/16 cities have lower predicted humidity
   📏 Avg difference: +20.3 %
   ✅ Conclusion: NOT drier air

💾 Saved → /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_hypotheses.csv


## §6 · Interactive wildfire risk map (Folium, date-selectable)

In [13]:
# Daily max risk per city for the map
fire_fc_copy = fire_fc.copy()
fire_fc_copy["Date"] = fire_fc_copy["Timestamp"].dt.date
daily_risk = (fire_fc_copy.groupby(["City", "Date"])
              .agg(max_risk=("fire_risk_prob", "max"),
                   alert_hours=("fire_risk_alert", "sum"))
              .reset_index())
daily_risk["Latitude"]  = daily_risk["City"].map(lambda c: CITIES.get(c, (40.4, 49.8))[0])
daily_risk["Longitude"] = daily_risk["City"].map(lambda c: CITIES.get(c, (40.4, 49.8))[1])

dates_risk = sorted(daily_risk["Date"].unique())

risk_map = folium.Map(location=[40.5, 47.5], zoom_start=7,
                      tiles="cartodbpositron", control_scale=True)

RISK_COLORS = {"Low": "#27ae60", "Moderate": "#f39c12",
               "High": "#e74c3c", "Extreme": "#8e44ad"}

for date in dates_risk:
    fg = folium.FeatureGroup(name=str(date), show=(date == dates_risk[0]))
    day_data = daily_risk[daily_risk["Date"] == date]

    for _, row in day_data.iterrows():
        risk_val = row["max_risk"]
        if   risk_val < 0.05: risk_label, color = "Low",      RISK_COLORS["Low"]
        elif risk_val < 0.20: risk_label, color = "Moderate",  RISK_COLORS["Moderate"]
        elif risk_val < 0.50: risk_label, color = "High",      RISK_COLORS["High"]
        else:                 risk_label, color = "Extreme",   RISK_COLORS["Extreme"]

        popup_html = (f"<b>{row['City']}</b> — {date}<br>"
                      f"Max fire risk: {risk_val:.1%}<br>"
                      f"Alert hours: {int(row['alert_hours'])}/24<br>"
                      f"Risk class: {risk_label}")

        folium.CircleMarker(
            location=[row["Latitude"], row["Longitude"]],
            radius=max(6, risk_val * 30),
            color=color, fill=True, fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{row['City']} — {risk_label} ({risk_val:.0%})",
        ).add_to(fg)

    # Heatmap layer for this date
    heat_data = [[r["Latitude"], r["Longitude"], r["max_risk"]]
                 for _, r in day_data.iterrows() if r["max_risk"] > 0.05]
    if heat_data:
        HeatMap(heat_data, radius=25, blur=15, max_zoom=10,
                name=f"Heat {date}").add_to(fg)

    fg.add_to(risk_map)

folium.LayerControl(collapsed=False).add_to(risk_map)

legend_html = """
<div style="position:fixed; bottom:20px; left:20px; z-index:1000;
            background:white; padding:10px; border:2px solid #999;
            border-radius:6px; font-size:12px;">
  <b>Wildfire Risk Level</b><br>
  <span style="color:#27ae60">&#9679;</span> Low (&lt;5%) &nbsp;
  <span style="color:#f39c12">&#9679;</span> Moderate (5–20%) &nbsp;
  <span style="color:#e74c3c">&#9679;</span> High (20–50%) &nbsp;
  <span style="color:#8e44ad">&#9679;</span> Extreme (&gt;50%)<br>
  <i>Use layer control (top-right) to switch dates</i>
</div>
"""
risk_map.get_root().html.add_child(folium.Element(legend_html))

risk_map.save(str(OUTPUTS / "phase4_risk_map.html"))
print(f"💾 Saved risk map → {OUTPUTS / 'phase4_risk_map.html'}")
risk_map

💾 Saved risk map → /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_risk_map.html


## §7 · Plotly risk dashboard

In [14]:
# Heatmap: City × Date with max risk probability
pivot_risk = daily_risk.pivot_table(index="City", columns="Date",
                                     values="max_risk", aggfunc="max")

fig = go.Figure(data=go.Heatmap(
    z=pivot_risk.values,
    x=[str(d) for d in pivot_risk.columns],
    y=pivot_risk.index.tolist(),
    colorscale=[[0, "#27ae60"], [0.2, "#f39c12"], [0.5, "#e74c3c"], [1, "#8e44ad"]],
    zmin=0, zmax=1, colorbar_title="Max Fire Risk",
))
fig.update_layout(title="30-Day Wildfire Risk Heatmap (City × Date)",
                  xaxis_title="Date", yaxis_title="City",
                  height=600, template="plotly_white")
fig.write_html(str(OUTPUTS / "phase4_risk_heatmap.html"))
fig.show()

# Per-city risk timeline
fig2 = make_subplots(rows=4, cols=4, shared_xaxes=True,
                     subplot_titles=sorted(CITIES.keys())[:16],
                     vertical_spacing=0.06)
city_list = sorted(CITIES.keys())[:16]
for idx, city in enumerate(city_list):
    r, c = divmod(idx, 4)
    city_risk = fire_fc[fire_fc["City"] == city].sort_values("Timestamp")
    fig2.add_trace(go.Scatter(
        x=city_risk["Timestamp"], y=city_risk["fire_risk_prob"],
        mode="lines", name=city, line=dict(width=1),
        showlegend=False,
    ), row=r+1, col=c+1)
    fig2.add_hline(y=op_thr, line_dash="dash", line_color="red",
                   opacity=0.4, row=r+1, col=c+1)

fig2.update_layout(height=900, template="plotly_white",
                   title="Hourly Fire Risk Probability per City (30-day)")
fig2.update_yaxes(range=[0, 1])
fig2.write_html(str(OUTPUTS / "phase4_city_timelines.html"))
fig2.show()

## §8 · Summary

In [15]:
print("=" * 70)
print("ARIAN PHASE 4 — PIPELINE COMPLETE")
print("=" * 70)

artifacts = {
    "Wildfire hourly 30d forecast": OUTPUTS / "phase4_wildfire_hourly_30d.parquet",
    "Test-set metrics":             OUTPUTS / "phase4_wildfire_scores.csv",
    "Feature importance":           OUTPUTS / "phase4_feature_importance.csv",
    "Hypothesis report":            OUTPUTS / "phase4_hypotheses.csv",
    "Risk map (Folium)":            OUTPUTS / "phase4_risk_map.html",
    "Risk heatmap (Plotly)":        OUTPUTS / "phase4_risk_heatmap.html",
    "City timelines (Plotly)":      OUTPUTS / "phase4_city_timelines.html",
    "XGBoost model":                MODELS / "phase4_xgb_fire.joblib",
    "Model manifest":               MODELS / "phase4_manifest.json",
}

for label, path in artifacts.items():
    status = "✅" if path.exists() else "❌ MISSING"
    print(f"  {status} {label}: {path}")

# High-risk city summary
top_risk_cities = (daily_risk.groupby("City")["max_risk"]
                   .max().sort_values(ascending=False).head(5))
print(f"\n🔥 Top-5 highest-risk cities in the 30-day window:")
for city, risk in top_risk_cities.items():
    print(f"   {city}: {risk:.1%}")

print("\n🏁 All done. Review the hypothesis report and risk maps above.")

ARIAN PHASE 4 — PIPELINE COMPLETE
  ✅ Wildfire hourly 30d forecast: /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_wildfire_hourly_30d.parquet
  ✅ Test-set metrics: /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_wildfire_scores.csv
  ✅ Feature importance: /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_feature_importance.csv
  ✅ Hypothesis report: /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_hypotheses.csv
  ✅ Risk map (Folium): /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_risk_map.html
  ✅ Risk heatmap (Plotly): /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_risk_heatmap.html
  ✅ City timelines (Plotly): /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/outputs/phase4_city_timelines.html
  ✅ XGBoost model: /home/manheim666/Desktop/ARIAN ALL FOLDERS/ARIAN_ASIF/models/phase4_xgb_fire.joblib
  ✅ Model manifest: /home/manheim666/Desktop/ARIAN ALL FOLDE